# Interactive dataset viewer

Slide through the image/mask pairs in `../dataset`; pick which classes to overlay and the blend alpha.

Kernel: **Python (endo)**. Deps: numpy, pillow, matplotlib, ipywidgets.

In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import ipywidgets as widgets

ROOT = os.path.abspath("..")
IMG_DIR = os.path.join(ROOT, "dataset", "image")
LBL_DIR = os.path.join(ROOT, "dataset", "label")

CLASS_NAMES = {0: "background"}
with open(os.path.join(ROOT, "dataset", "label distribution.txt")) as f:
    for line in f:
        line = line.strip()
        if line:
            idx, name = line.split(" ", 1)
            CLASS_NAMES[int(idx)] = name
N_CLASSES = max(CLASS_NAMES) + 1
COLORS = plt.get_cmap("tab10")(np.arange(N_CLASSES))[:, :3]

stems = sorted(p[:-4] for p in os.listdir(IMG_DIR) if p.endswith(".jpg"))
print(len(stems), "pairs")


def load(stem):
    img = np.array(Image.open(os.path.join(IMG_DIR, stem + ".jpg")).convert("L"))
    mask = np.array(Image.open(os.path.join(LBL_DIR, stem + ".png")))  # palette-indexed -> 2D
    return img, mask


def overlay(img, mask, alpha=0.5):
    rgb = np.stack([img] * 3, -1).astype(float) / 255
    for c in range(1, N_CLASSES):
        m = mask == c
        rgb[m] = (1 - alpha) * rgb[m] + alpha * COLORS[c]
    return np.clip(rgb, 0, 1)


LEGEND = [Patch(facecolor=COLORS[c], label=CLASS_NAMES[c]) for c in range(1, N_CLASSES)]

In [ ]:
def view(i, alpha, classes):
    stem = stems[i]
    img, mask = load(stem)
    shown = np.where(np.isin(mask, classes), mask, 0)  # zero out unselected classes
    present = [int(v) for v in np.unique(mask) if v]
    fig, ax = plt.subplots(1, 2, figsize=(14, 7))
    ax[0].imshow(img, cmap="gray"); ax[0].set_title(stem)
    ax[1].imshow(overlay(img, shown, alpha))
    ax[1].set_title("present: " + ", ".join(CLASS_NAMES[c] for c in present))
    for a in ax:
        a.axis("off")
    fig.legend(handles=LEGEND, loc="lower center", ncol=5, fontsize=8)
    fig.tight_layout(rect=[0, 0.06, 1, 1])
    plt.show()


widgets.interact(
    view,
    i=widgets.IntSlider(value=0, min=0, max=len(stems) - 1, step=1, description="sample",
                        continuous_update=False),  # don't reload the jpg mid-drag
    alpha=widgets.FloatSlider(value=0.5, min=0, max=1, step=0.05, description="alpha"),
    classes=widgets.SelectMultiple(
        options=[(CLASS_NAMES[c], c) for c in range(1, N_CLASSES)],
        value=tuple(range(1, N_CLASSES)), rows=9, description="classes"),
);